# Buzzards Bay basin rainfall dashboard

Pick a basin and a start/stop date-time to see the **average daily rainfall over that basin**. Each point is the area-weighted mean depth over the basin for the 24 h ending 7am US Eastern on that date (NOAA MRMS 1 km radar/gauge estimates), so the time filter applies to that 7am window end.

Run the last cell to get the dashboard inline, or serve this notebook as an app with `panel serve basin_dashboard.ipynb --show`.

Data is public, no credentials: the daily time series from Cloudflare R2 (rebuilt daily by `run_daily.sh`) and the basin names from the v2 geojson on GitHub.

In [1]:
# %pip install -q hvplot holoviews panel pyogrio pyarrow   # uncomment if a package is missing

In [2]:
import holoviews as hv
import hvplot.pandas  # noqa: F401  (registers .hvplot)
import pandas as pd
import panel as pn
from pyogrio import read_dataframe

pn.extension()
hv.extension("bokeh")

ERROR 1: PROJ: proj_create_from_database: Open of /home/rsignell/miniforge3/envs/protocoast-notebook/share/proj failed


## Data

Fetched once and cached for 3 hours.

In [3]:
POLYGONS = ("https://raw.githubusercontent.com/rsignell/buzzards-bay-precip/"
            "main/basins/bbnep_subbasins_2026_v2.geojson")
SERIES = "https://r2-pub.openscicomp.io/buzzards-bay-precip/basins_v2_precip_ts.parquet"
# Cloudflare rejects Python's default User-Agent (HTTP 403).
UA = {"User-Agent": "buzzards-bay-precip-dashboard"}
BLUE, RED = "#2563eb", "#dc2626"


@pn.cache(max_items=1, ttl=3 * 3600)
def load_data():
    meta = read_dataframe(POLYGONS, read_geometry=False)      # names only, no geometry
    meta["basin_id"] = meta["BBP_SYS_ID"] + "_" + meta["TYPE"]
    ts = pd.read_parquet(SERIES, storage_options=UA)
    ts["window_end"] = ts["report_date"] + pd.Timedelta(hours=7)   # 7am local, naive
    return meta.set_index("basin_id"), ts


meta, ts = load_data()
names = meta["TMDL_BASIN"]
earliest, latest = ts["window_end"].min(), ts["window_end"].max()
print(f"{ts.basin_id.nunique()} basins, {ts.report_date.nunique():,} report dates, "
      f"{earliest:%Y-%m-%d} to {latest:%Y-%m-%d}")

66 basins, 4,340 report dates, 2014-11-02 to 2026-09-19


## Widgets

In [4]:
def _group(kind):
    m = meta[meta["TYPE"] == kind].sort_values("TMDL_BASIN")
    return {row.TMDL_BASIN: bid for bid, row in m.iterrows()}


basin_w = pn.widgets.Select(
    name="Basin",
    groups={"Water (embayments and open bay)": _group("WATER"),
            "Land (drainage areas)": _group("LAND")},
    value="BBNEMB_WATER")

range_w = pn.widgets.DatetimeRangePicker(
    name="Start and stop (US Eastern)",
    value=((latest - pd.Timedelta(days=365)).to_pydatetime(), latest.to_pydatetime()),
    start=earliest.to_pydatetime(), end=latest.to_pydatetime(),
    enable_time=True, enable_seconds=False, military_time=True)

NOTE = f"""
Each point is the **average rainfall over the basin** for the 24 h ending
7am US Eastern on that date, from NOAA MRMS 1 km radar/gauge estimates.

Red dots mark days with missing MRMS hours; those totals are lower bounds
(or gaps if there is no data at all). The record starts {earliest:%Y-%m-%d}
and Nov 2014 to 2015 has many gaps.
"""

## Plot

A summary line plus an hvplot time series for the chosen basin and range. Empty ranges show a message instead of a plot.

In [5]:
def view(basin_id, dt_range):
    if not dt_range:
        return pn.pane.Alert("Pick a start and a stop date and time.", alert_type="info")
    start, stop = (pd.Timestamp(t) for t in dt_range)

    d = ts[(ts["basin_id"] == basin_id) & ts["window_end"].between(start, stop)]
    if d["precip_mm"].notna().sum() == 0:
        return pn.pane.Alert(
            f"No MRMS data for {names[basin_id]} between {start:%Y-%m-%d %H:%M} and "
            f"{stop:%Y-%m-%d %H:%M}. The record runs {earliest:%Y-%m-%d} to {latest:%Y-%m-%d}.",
            alert_type="warning")

    partial = d[d["n_hours"] < d["n_hours_expected"]]
    wet = d.loc[d["precip_mm"].idxmax()]
    total_mm = d["precip_mm"].sum()
    summary = (
        f"**{names[basin_id]}** ({meta.loc[basin_id, 'TYPE'].lower()}), "
        f"{start:%Y-%m-%d %H:%M} to {stop:%Y-%m-%d %H:%M}  \n"
        f"Mean **{d['precip_mm'].mean():.2f} mm/day** ({d['precip_in'].mean():.3f} in/day) "
        f"· total {total_mm:,.0f} mm ({total_mm / 25.4:,.1f} in) "
        f"· wettest {wet['report_date']:%Y-%m-%d} ({wet['precip_mm']:.1f} mm) "
        f"· {len(d):,} days, {len(partial):,} with missing hours")

    common = dict(x="window_end", y="precip_mm", responsive=True, height=460, xlabel="",
                  ylabel="average daily rainfall (mm)",
                  title=f"{names[basin_id]}: average daily rainfall",
                  hover_cols=["precip_in", "n_hours", "n_hours_expected"])
    plot = d.hvplot.line(color=BLUE, line_width=1.5, label="daily mean", **common)
    if len(partial):
        plot = (plot * partial.hvplot.scatter(color=RED, size=28, label="missing hours (lower bound)",
                                              **common)
                ).opts(legend_position="top_right")     # Overlay-only option
    return pn.Column(pn.pane.Markdown(summary),
                     pn.pane.HoloViews(plot, sizing_mode="stretch_width"),
                     sizing_mode="stretch_width")

## Dashboard

In [6]:
dashboard = pn.Row(
    pn.Column(basin_w, range_w, pn.pane.Markdown(NOTE, width=300), width=330),
    pn.Column(pn.bind(view, basin_w, range_w), sizing_mode="stretch_width"),
    sizing_mode="stretch_width")
dashboard.servable(title="Buzzards Bay basin rainfall")

Row(sizing_mode='stretch_width')
    [0] Column(width=330)
        [0] Select(groups={'Water (embayments and op...}, label='Basin', name='Basin', value='BBNEMB_WATER')
        [1] DatetimeRangePicker(as_numpy_datetime64=False, enable_seconds=False, end=datetime.datetime(2026, ..., label='Start and stop (..., name='Start and stop (..., start=datetime.datetime(2014, ..., value=(datetime.datetime(2025, ...)
        [2] Markdown(str, width=300)
    [1] Column(sizing_mode='stretch_width')
        [0] ParamFunction(function, _pane=Column, defer_load=False)